# Lesson 0007: attention, looking further back than one token

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tamnd/soroban/blob/main/lessons/0007-attention/lesson.ipynb)

Every model so far has had a one-token memory: the bigram and the neural bigram both predict the next letter from the current letter alone. Attention removes that wall. It lets each position read every earlier position, decide from the data how much each matters, and mix them. This notebook runs one causal head by hand, counts the two loss floors that show why a bigram is stuck, then trains a tiny one-head transformer on a few thousand characters of TinyStories. The full writeup is in the [lesson README](https://github.com/tamnd/soroban/tree/main/lessons/0007-attention).

![one causal head over three tokens](https://raw.githubusercontent.com/tamnd/soroban/main/lessons/0007-attention/assets/pattern.png)

## 0. One attention head over three tokens

Give three tokens the fixed embeddings `x0 = (1, 0)`, `x1 = (0, 1)`, `x2 = (1, 1)`, and set the query, key, and value projections to the identity so every number is forced. The score of position j for position i is the dot product `x[i].x[j]` divided by `sqrt(d)`, the allowed scores go through a [softmax](https://github.com/tamnd/soroban/blob/main/maths/softmax.md), and the output is the weighted sum of the earlier embeddings. Causal means position i attends only to positions at or before it. The full mechanism is on the [attention page](https://github.com/tamnd/soroban/blob/main/maths/attention.md).

In [1]:
import numpy as np, math

E = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])

def softmax(scores):
    s = scores - scores.max()
    e = np.exp(s)
    return e / e.sum()

def head(embeddings):
    n, d = embeddings.shape
    scale = 1.0 / math.sqrt(d)
    weights, out = [], []
    for i in range(n):
        s = np.array([embeddings[i] @ embeddings[j] for j in range(i + 1)]) * scale
        w = softmax(s)
        o = sum(w[j] * embeddings[j] for j in range(i + 1))
        weights.append(w); out.append(o)
    return weights, out

weights, out = head(E)
for i in range(3):
    print(f"tok{i}  weights", np.round(weights[i], 6), " out", np.round(out[i], 6))
assert np.allclose(weights[2], [0.248255, 0.248255, 0.503490], atol=1e-6)
assert np.allclose(out[2], [0.751745, 0.751745], atol=1e-6)

tok0  weights [1.]  out [1. 0.]
tok1  weights [0.330238 0.669762]  out [0.330238 0.669762]
tok2  weights [0.248255 0.248255 0.50349 ]  out [0.751745 0.751745]


## 1. Why one letter is not enough: the two floors

Take the corpus aba, cbc, wrapped in the [boundary token](https://github.com/tamnd/soroban/blob/main/maths/notation.md). After b comes a in one word and c in the other, so a bigram keyed on b guesses 50/50 and pays `log 2 = 0.693147` at every position: that is the best a bigram can do. A model that could see the letter two back would be uncertain only at the first letter of each word, for a loss of `log(2)/4 = 0.173287`. The gap is the value of the context a bigram throws away.

In [2]:
def wrap(w): return "." + w + "."

def bigram_floor(corpus):
    nxt = {}
    for w in corpus:
        s = wrap(w)
        for a, b in zip(s, s[1:]):
            nxt.setdefault(a, {}).setdefault(b, 0); nxt[a][b] += 1
    tot, n = 0.0, 0
    for w in corpus:
        s = wrap(w)
        for a, b in zip(s, s[1:]):
            row = nxt[a]; tot += -math.log(row[b] / sum(row.values())); n += 1
    return tot / n

def context_floor(corpus):
    cont = {}
    for w in corpus:
        s = wrap(w)
        for k in range(1, len(s)):
            cont.setdefault(s[:k], {}).setdefault(s[k], 0); cont[s[:k]][s[k]] += 1
    tot, n = 0.0, 0
    for w in corpus:
        s = wrap(w)
        for k in range(1, len(s)):
            row = cont[s[:k]]; tot += -math.log(row[s[k]] / sum(row.values())); n += 1
    return tot / n

bg, cx = bigram_floor(["aba", "cbc"]), context_floor(["aba", "cbc"])
print("bigram floor ", round(bg, 6), " = log2   ", round(math.log(2), 6))
print("context floor", round(cx, 6), " = log2/4 ", round(math.log(2)/4, 6))
assert abs(bg - math.log(2)) < 1e-12 and abs(cx - math.log(2)/4) < 1e-12

bigram floor  0.693147  = log2    0.693147
context floor 0.173287  = log2/4  0.173287


## 2. Exit test: what the scale does

Recompute token 2's weights with no scale, dividing the raw dot products `1, 1, 2` by 1 instead of `sqrt(2)`. The softmax comes out sharper, `0.211942, 0.211942, 0.576117`: more mass on the strongest match, less on the ties. The `1/sqrt(d)` scale exists to hold that sharpening back, so that at large d the softmax does not collapse to one-hot and starve the gradient.

In [3]:
raw = np.array([1.0, 1.0, 2.0])   # unscaled dot products for token 2
unscaled = softmax(raw)
print("unscaled weights:", np.round(unscaled, 6))
print("scaled weights:  ", np.round(weights[2], 6))
assert np.allclose(unscaled, [0.211942, 0.211942, 0.576117], atol=1e-6)

unscaled weights: [0.211942 0.211942 0.576117]
scaled weights:   [0.248255 0.248255 0.50349 ]


## 3. The experiment: a few thousand characters of TinyStories

The training text is 19 short children's stories, lowercased and reduced to letters, spaces, and a little punctuation: 9311 characters, an alphabet of 33. First the baseline. A bigram counted on this exact text scores an average cross-entropy of 2.1651, the number the attention model has to beat.

In [4]:
CORPUS = """u don't have to be scared of the loud dog, i'll protect you . the mole felt so safe with the little girl. she was very kind and the mole soon came to trust her. he leaned against her and she kept him safe. the mole had found his best friend.
once upon a time, there was a wealthy man named tom. he had a big house near a cliff. tom liked to sort his many toys into different boxes. one sunny day, tom went outside to play with his toys. he took them all out of their boxes and spread them on the ground. he had fun playing with his cars, dolls, and balls. when it was time to go home, tom sorted his toys back into their boxes. he was happy to live in his big house near the cliff. and every day, he played with his toys and sorted them again and again.
once upon a time, there was a cool cat named tom. tom loved to go for a jog in the park. every day, he would put on his cool hat and go for a run. one sunny day, as tom was jogging, he saw a big tree. he decided to turn right and run around it. as he turned, he met a new friend, a dog named sam. sam was also going for a jog in the park. tom and sam jogged together every day. they would turn around the big tree, then sit under it to rest. they became best friends and had lots of fun in the cool park.
once upon a time, there was a dog named spot. spot was a very persistent dog. he loved to play and have fun. one day, spot heard his friends talking. we will celebrate! said one friend. spot was excited. he wanted to celebrate too. he ran to his friends and asked, can i celebrate too? his friends smiled and said, yes, spot! let's all celebrate together! they played games, ate yummy food, and laughed a lot. spot was very happy. his friends were happy too. they all had a great time celebrating. and they all lived happily ever after.
one day, a little girl named lily went to the park. she saw a pretty angel playing. the angel had big wings and a nice smile. lily wanted to catch the angel, but she was too slow. lily called out, angel, please wait for me! but the angel did not hear her. the angel was deaf. she could not hear anything. lily felt sad, but she had an idea. lily picked up a flower and threw it to the angel. the angel saw the flower and smiled at lily. she flew down to lily and they became friends. they played in the park all day, and lily was happy.
once upon a time, there was a long string. this string was very special. it lived in a big, pretty box. the string was very happy in the box. one day, a little boy found the box. he wanted to study the long string. he took the string out of the box and played with it. the string was very happy to be with the little boy. the little boy and the string became best friends. they played all day and had lots of fun. the long string was very happy to have a friend. and they lived happily ever after.
once upon a time, there was a polite bee named bob. bob lived in a big hive with all his bee friends. the hive was in a tall tree, near pretty flowers. bob loved his home. every day, bob and his friends went to the flowers to get food. they took the food back to the hive to store it. they worked together and shared with each other. one day, bob met a new friend, a butterfly named bella. bella was very nice and polite too. they played together and had lots of fun. from that day on, bob and bella were the best of friends.
once upon a time, there was a little beetle named bob. bob was very popular. all his friends liked him a lot. bob lived in a big green tree. one day, bob was playing with his friends when a new beetle appeared. the new beetle was shy and said, hi, i'm tim. can i play too? bob and his friends were happy to have a new friend. they all played together and had so much fun. tim was happy to be with bob and his friends. now, tim was popular too. they all lived happily in the big green tree.
once upon a time, there was a pretty flower. the flower lived in a big garden. one day, a little boy named tim saw the flower. he liked it a lot. tim said, i want to test if the flower can be mine. he picked the flower from the ground. but, oh no! the flower was spoiled. the pretty flower turned brown and sad. tim cried and told his mom, i picked the flower and it got spoiled. his mom said, you should not pick flowers, they are happy in the garden. tim learned that it is better to leave pretty things where they are.
once upon a time, there was a gifted bird named blue. blue could whistle the best songs in the forest. all the other animals loved to hear blue whistle. one day, blue found a shiny black rock. it was coal. blue took the coal to his friend, bunny. bunny liked the coal and used it to draw pictures on the ground. blue and bunny had a fun day playing with the coal and whistling songs. all the animals in the forest came to see the pictures and hear blue whistle. they all had a great time together.
once upon a time, there was an adorable little dog named max. max loved to play with his gear. he had a ball, a bone, and a rope. max played with his gear all day long. one day, max saw a big cat. the cat said, bow to me, little dog. max did not want to bow to the cat, but he did it anyway. the cat laughed and took max's gear. max went home without his gear. he was very sad. his owner tried to make him happy, but max missed his gear too much. the cat never gave max his gear back, and max stayed sad.
once upon a time, in a small house, there was a little girl named mia. mia had a pet bird named bob. bob lived in a cage. mia loved bob very much. one day, mia saw that bob was sad. she asked, bob, why are you sad? bob said, i want to be free and fly. mia felt sad for bob. she wanted to make bob happy. mia opened the cage door and let bob out. bob was very happy. he flew around the room. mia was proud of her bird. she said, bob, i love you! bob flew to mia and let her rub his head. they were both happy and played together all day.
one day, a popular cat named tom went for a walk. he saw a jar on the side of the road. tom was curious, so he stepped closer to look at it. hey, tom, said a bird named sue. what's in the jar? tom didn't know, so he opened the jar. out jumped a tiny frog! they were both surprised. thank you for letting me out, said the frog. i will give you a wish. tom and sue looked at each other. they wished to be friends forever. the frog smiled and their wish came true. they were all very happy.
one day, a boy named tim was very excited. his mom and dad were going to send him to the zoo. he had never been to the zoo before. tim could not wait to see all the animals. when they got to the zoo, tim saw a big lion. the lion roared loud. he also saw a tall giraffe with a long neck. the giraffe ate leaves from the tree. tim was so happy to see all the animals. at the end of the day, tim and his mom and dad went home. tim was very tired but still excited. he told all his friends about the zoo. tim could not wait to go back to the zoo again.
once upon a time, in a peaceful town, there was a big square. in the square, there were many kids who liked to play. they were very happy. one day, a little girl saw a puzzle on the ground. it had many square pieces. she wanted to solve it. so, she asked her friends to help her. they all worked together to solve the puzzle. they put the square pieces in the right place. soon, the puzzle was done. the kids were so happy and proud. they had a fun day in the peaceful town.
one day, a cat and a dog were in a park. the cat was comfortable on a bench. the dog was near a grill. the dog said, i want to play! the cat looked at the dog and said, okay, let's play! they played near the grill. the dog ran fast and the cat jumped high. they had fun. then, the dog's tail hit the grill. ouch! said the dog. the cat ran to help. the cat gave the dog a soft slap on the back. the dog felt better. they went back to play and had a great day.
one day, tim went for a walk with his mom. they saw a big pile of junk near their house. tim was very alert and observed something shiny in the junk. mom, look! tim said. i see something shiny. can i get it? his mom said, okay, but be careful. tim carefully moved the junk and found a toy car. he was so happy. he showed the car to his mom, and she smiled. from then on, tim always observed his surroundings and found many more treasures. he learned that being alert can lead to finding special things.
once upon a time, there was a little boat. the boat liked to go to the shore. one day, the boat saw a big load. the load was heavy and uncomfortable. the boat wanted to help. so, the boat took the load to the shore. the load made the boat very uncomfortable. the boat felt slow and tired. in the end, the boat could not carry the load anymore. the boat stopped moving and stayed on the shore. the boat was sad and uncomfortable forever.
once upon a time, a kind girl named lily went for a walk. she saw a pretty purse on the ground. lily liked the purse and wanted to find who it belonged to. lily asked her friends, do you know who lost this purse? her friends did not know, but they all admired the purse. they thought it was very nice. finally, lily found a sad lady who had lost her purse. the lady was so happy when lily gave it back to her. the lady said, thank you, kind girl! lily smiled and felt good for being helpful."""

chars = sorted(set(CORPUS)); V = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
ids = [stoi[c] for c in CORPUS]
print(f"{CORPUS.count(chr(10)) + 1} stories, {len(CORPUS)} chars, vocab {V}")

N = np.zeros((V, V))
for a, b in zip(ids, ids[1:]): N[a, b] += 1
P = N / np.clip(N.sum(1, keepdims=True), 1, None)
baseline = np.mean([-math.log(max(P[a, b], 1e-12)) for a, b in zip(ids, ids[1:])])
print("bigram baseline:", round(baseline, 4))
assert 2.0 < baseline < 2.3

19 stories, 9311 chars, vocab 33
bigram baseline: 2.1651


## 4. A tiny transformer, trained

The smallest model that still deserves the name: a token embedding, a position embedding, one attention head, a small MLP, and a linear head, about 58 thousand weights. Train it for three thousand steps, about a minute on a CPU. Its loss starts above the bigram and settles far below, around 0.38, because it can use context the bigram cannot see. (This cell needs torch, which Colab has preinstalled.)

In [5]:
try:
    import torch, torch.nn.functional as F

    torch.manual_seed(1337)
    data = torch.tensor(ids, dtype=torch.long)
    B, T, D = 32, 64, 64
    n = len(data)

    class Block(torch.nn.Module):
        def __init__(self):
            super().__init__()
            self.tok = torch.nn.Embedding(V, D); self.pos = torch.nn.Embedding(T, D)
            self.q = torch.nn.Linear(D, D, bias=False)
            self.k = torch.nn.Linear(D, D, bias=False)
            self.v = torch.nn.Linear(D, D, bias=False)
            self.proj = torch.nn.Linear(D, D)
            self.ln1 = torch.nn.LayerNorm(D); self.ln2 = torch.nn.LayerNorm(D)
            self.mlp = torch.nn.Sequential(
                torch.nn.Linear(D, 4 * D), torch.nn.GELU(), torch.nn.Linear(4 * D, D))
            self.lnf = torch.nn.LayerNorm(D); self.head = torch.nn.Linear(D, V)
            self.register_buffer("mask", torch.tril(torch.ones(T, T)))

        def forward(self, idx):
            Tt = idx.shape[1]
            x = self.tok(idx) + self.pos(torch.arange(Tt))
            h = self.ln1(x)
            att = (self.q(h) @ self.k(h).transpose(-2, -1)) / math.sqrt(D)
            att = att.masked_fill(self.mask[:Tt, :Tt] == 0, float("-inf"))
            att = F.softmax(att, dim=-1)
            x = x + self.proj(att @ self.v(h))
            x = x + self.mlp(self.ln2(x))
            return self.head(self.lnf(x))

    model = Block()
    opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
    nparams = sum(p.numel() for p in model.parameters())
    for _ in range(3000):
        ix = torch.randint(0, n - T - 1, (B,))
        x = torch.stack([data[i:i+T] for i in ix])
        y = torch.stack([data[i+1:i+T+1] for i in ix])
        loss = F.cross_entropy(model(x).reshape(-1, V), y.reshape(-1))
        opt.zero_grad(); loss.backward(); opt.step()
    print(f"{nparams} params, 3000 steps: train loss {loss.item():.4f}  (bigram was {baseline:.4f})")
    assert loss.item() < 1.0

    @torch.no_grad()
    def sample(nchars=220):
        ctx = [stoi["\n"]]; out = []
        for _ in range(nchars):
            idx = torch.tensor([ctx[-T:]])
            p = F.softmax(model(idx)[0, -1], dim=-1)
            nx = torch.multinomial(p, 1).item()
            out.append(itos[nx]); ctx.append(nx)
        return "".join(out).replace("\n", " ")

    print("\nsample:", sample())
except ImportError:
    print("torch not installed, skipping the training cell (Colab has it preinstalled)")

58273 params, 3000 steps: train loss 0.3906  (bigram was 2.1651)

sample: levery popened again. once upon a tid a tall tree, new beetle named bob. bob lived his home. every alert and asked, being a's afelew with the load to be free. the little dog named spot. spot heared that it lis bee friend


## 5. The head, cross-checked in torch

The same three-token head built with `masked_fill` and `softmax` gives token 2 the same output, `(0.751745, 0.751745)`: the arithmetic is the mechanism, not a numpy accident.

In [6]:
try:
    import torch

    Et = torch.tensor(E)
    scale = 1.0 / math.sqrt(Et.shape[1])
    scores = (Et @ Et.T) * scale
    mask = torch.tril(torch.ones(3, 3))
    scores = scores.masked_fill(mask == 0, float("-inf"))
    w = torch.softmax(scores, dim=1)
    o = w @ Et
    print("torch tok2 out:", np.round(o[2].numpy(), 6))
    assert torch.allclose(o[2], torch.tensor([0.751745, 0.751745], dtype=torch.float64), atol=1e-6)
    print("torch agrees with the by-hand head")
except ImportError:
    print("torch not installed, skipping (Colab has it preinstalled)")

torch tok2 out: [0.751745 0.751745]
torch agrees with the by-hand head


## Exercises

1. Which of the three tokens has the sharpest attention, the one concentrating the most weight on a single position? Predict, then read the numbers.
2. The bigram floor is `log 2` and the context floor is `log(2)/4`. Explain in one sentence why the ratio is exactly four, using the count of positions and how many are ambiguous.
3. The training loss lands far below the bigram, but on nine thousand characters with sixty thousand weights the model mostly memorizes. What single number, not computed here, would tell you how much it actually generalizes?

Worked answers are in the [lesson README](https://github.com/tamnd/soroban/tree/main/lessons/0007-attention) and asserted in `train.py`. Lesson 0008 turns the training loop itself into an instrument, measuring exactly the generalization gap exercise 3 asks about.